# Stage 3b: the checkpoint report

**Reads:** `results/checkpoint/inputs/dev_runs.csv` and `results/checkpoint/inputs/dev_components.csv`, unchanged copies of the two files of part 1 of stage 3 committed from Nuvolos. With `ADA_CHECKPOINT=rerun`, the copies of the rerun of part 1 in `results/checkpoint/rerun/inputs/` instead.

**Writes:** into `results/checkpoint/` (or `results/checkpoint/rerun/`): one to three `.csv` files for each item of the report, `learning_curves.pdf`, and `rules.csv`, what the rules of the checkpoint call for. `results/README.md` lists them.

It is not in the order of a full run. It runs between part 1 and the full run of stage 3, with `python code/run_all.py --only 03b_checkpoint.ipynb`, and once more on the rerun of part 1 if the checkpoint calls for one; quick mode runs it after stage 3, so that every push tests it. It fits nothing. The report has the nine items of section 6 of `docs/build_plan.md`, for every size of part 1. The functions behind it are in `helpers.py`, and stages 4, 5 and 7 use the same ones, so the checkpoint and the final tables cannot disagree on the same sizes.

Every standard error here describes variation over training draws from one pool, scored on one test set. The draws overlap, more so at large sizes.

*Drafted with Claude (Anthropic), which was asked for the checkpoint report of the build plan: the learning curves, the scores, the selection table, the gaps, the tuned settings and their grid edges, where the error comes from, the health checks, the time and its projection, the coefficients of `ppml`, and what the rules of the checkpoint call for. Later asked for (21 October 2026, decision 19, "Apply"; prepared on 20 October before Yu's answer): the text of items 5 and 8 after the boosting grid was extended, and item 5 read on the grid its inputs were made on.*

In [ ]:
from helpers import (CANNOT_EXTEND, CHECKPOINT, DRAWS, GRID_LIMIT, LINES, ROOT, SECTION9_STAGE_MINUTES, SIZES,
                     better_ensemble, check_find_crossing, crossings, deviance_by_group, divergences, edges, gaps,
                     health_table, learning_curves, line_configs, per_draw, ppml_coefficients, ppml_wins,
                     projected_seconds, refuse_part, seconds_per_draw, section9_seconds, selection_choices,
                     selection_confusion, size_rule, summarise, thin_indicators, tune, worse_than_mean)

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

refuse_part()
# Section 8: find_crossing on toy gaps, before it is used on real ones.
check_find_crossing()

INPUTS = CHECKPOINT / "inputs"
RERUN = os.environ.get("ADA_CHECKPOINT") == "rerun"
# round_trip reads every number back exactly as stage 3 wrote it.
runs = pd.read_csv(INPUTS / "dev_runs.csv", float_precision="round_trip")
components = pd.read_csv(INPUTS / "dev_components.csv", float_precision="round_trip")
sizes = sorted(int(n) for n in runs["size"].unique())
print(f"Inputs: {INPUTS.relative_to(ROOT)}" + (" (the rerun of part 1)" if RERUN else ""))
print(f"Sizes: {sizes}; {runs['draw'].nunique()} draws; {runs['config'].nunique()} configurations")


def save(table, name):
    # Every table of the report goes to the checkpoint folder, next to its inputs.
    table.to_csv(CHECKPOINT / name, index=False, lineterminator="\n")

## Before the report: the inputs are complete

Every size has the same number of draws for every configuration, and every test deviance is finite. Stage 3 checks that every prediction is positive and finite and writes no file if one is not, so a failure here means the inputs are not the files of part 1.

In [ ]:
draws_per = runs.groupby(["size", "config"])["draw"].nunique()
assert draws_per.nunique() == 1, "Some size and configuration has a different number of draws."
assert np.isfinite(runs["test_deviance"]).all(), "A test deviance in the inputs is not finite."
print(f"{len(runs):,} rows: {draws_per.iloc[0]} draws for each of {len(draws_per):,} sizes and configurations; every test deviance is finite")

summary = summarise(runs)
tuned = tune(summary)
lines = line_configs(tuned)

## Item 1. Learning curves

The mean test deviance of `ppml`, tuned `rf`, tuned `gbm`, `mean` and `tree` at each size, with a band of two standard errors, and their mean training deviance as dashed lines (in-sample for `ppml`, `gbm` and `mean`; out-of-bag for `rf`; none for `tree`, which fits its own sample exactly). Tuned means the setting with the lowest mean test deviance at that size (`tune`), so a dashed line can jump where the tuned setting changes. The space between a model's two lines is its optimism: how much better it fits its own sample than new pairs. For `rf` it is understated, because its dashed line is out-of-bag rather than in-sample: each training row is scored only by the trees that did not see it. The other direction of that pair can still be in those trees' samples, which lowers the dashed line a little.

Both axes are logarithmic. Where a band reaches below zero, it is drawn down to the bottom of the plot. The numbers behind the figure are in `learning_curves.csv`.

In [ ]:
curves = learning_curves(summary, lines)
save(curves, "learning_curves.csv")

# Categorical colours in a fixed order, one per line; each line also has its own marker.
COLOURS = {"ppml": "#2a78d6", "rf": "#eb6834", "gbm": "#1baf7a", "mean": "#eda100", "tree": "#e87ba4"}
MARKERS = {"ppml": "o", "rf": "s", "gbm": "^", "mean": "D", "tree": "v"}
LABELS = {"ppml": "ppml", "rf": "rf, tuned", "gbm": "gbm, tuned", "mean": "mean (reference)", "tree": "tree (reference)"}

positive = pd.concat([curves["mean"], curves["train_deviance"].dropna(), curves.loc[curves["lower"] > 0, "lower"]])
bottom = positive.min() / 2
figure, axes = plt.subplots(figsize=(9, 5))
for line in LINES:
    rows = curves[curves["line"] == line].sort_values("size")
    axes.fill_between(rows["size"], rows["lower"].clip(lower=bottom), rows["upper"], color=COLOURS[line], alpha=0.12, linewidth=0)
    axes.plot(rows["size"], rows["mean"], color=COLOURS[line], marker=MARKERS[line], markersize=6, linewidth=2, label=LABELS[line])
    if rows["train_deviance"].notna().any():
        axes.plot(rows["size"], rows["train_deviance"], color=COLOURS[line], marker=MARKERS[line], markersize=4,
                  linewidth=1.5, linestyle="--")
axes.set_xscale("log")
axes.set_yscale("log")
axes.set_ylim(bottom=bottom)
axes.set_xticks(sizes)
axes.set_xticklabels([f"{n:,}" for n in sizes])
axes.minorticks_off()
axes.set_xlabel("Training size (rows of country pairs)")
axes.set_ylabel("Mean Poisson deviance (thousand USD)")
axes.set_title("Test deviance (solid, two standard errors shaded) and training deviance (dashed)", fontsize=10)
axes.grid(True, which="major", color="#e5e5e2", linewidth=0.8)
axes.legend(frameon=False, fontsize=9, loc="upper left", bbox_to_anchor=(1.01, 1))
figure.tight_layout()
# No creation date in the file, so that the same inputs give the same file.
figure.savefig(CHECKPOINT / "learning_curves.pdf", metadata={"CreationDate": None})
plt.show()

## Item 2. Scores

For the same five lines: the mean test deviance with its standard error, the median, the largest draw and the share of the sum over draws that comes from it, D squared (mean and median over draws: the share of the deviance of predicting the mean test flow that the model removes), and the mean absolute error and root mean squared error, each divided by the mean test flow. Where one draw holds more than half of the sum, its standard error is marked unreliable. The mean test deviance is the score that decides (decision 12); the rest is there to be read.

In [ ]:
scores = summary.merge(lines, on=["size", "config"])
scores = scores[["size", "line", "config", "draws", "mean", "se", "se_unreliable", "median", "largest", "largest_share",
                 "d2_mean", "d2_median", "mae_rel", "rmse_rel"]]
save(scores, "scores.csv")
with pd.option_context("display.width", 200, "display.max_columns", None):
    print(scores.drop(columns="config").to_string(index=False, float_format="{:.4g}".format))

## Item 3. Selection at the middle point

At each size, the size rule picks the candidate with the lowest mean test deviance among `ppml`, tuned `rf` and tuned `gbm` (`size_rule`). On each draw, the winner is the one of the three with the lowest test deviance on that draw. The table counts, for each size, the draws each model won, under the model the rule picked; pooled over sizes it is a table of picked model against winner. Beside it: the share of draws picked right, the share that always picking the most frequent winner of that size would get, the mean regret (the picked model's test deviance minus the lowest of the three on the same draw) with its paired standard error, and the number of draws in which `ppml` beats tuned `rf`, tuned `gbm` and the ensemble of the `best` comparison (the better of the two at that size).

In the development year the pick and the winner are read on the same test set, so these numbers are optimistic. They preview H2, which is tested only in the analysis year.

In [ ]:
draws = per_draw(runs, lines)
choices = selection_choices(draws, size_rule(summary, lines))
counts, selection = selection_confusion(choices)
wins = ppml_wins(draws, better_ensemble(tuned))
save(counts, "selection_counts.csv")
save(selection, "selection_scores.csv")
save(wins, "ppml_wins.csv")
print(counts.to_string(index=False))
print()
print(selection.to_string(index=False, float_format="{:.4g}".format))
print()
print("Draws in which ppml has the lower test deviance (best: the better of tuned rf and tuned gbm):")
print(wins.to_string(index=False))

## Item 4. Gaps, provisional

For `ppml` against tuned `rf`, against tuned `gbm` and against the better of the two (`best`): the error gap E(ppml) - E(c) with its paired standard error, whether `ppml` is within two standard errors of the ensemble, and the bias gap and variance gap of stage 4 in their three versions (raw, r and rn, section 7). A negative error gap means `ppml` has the lower mean test deviance. Then `find_crossing` on the sizes so far, for each comparison. Both are provisional: the full run adds the larger sizes, and the development-year n* is exploratory in any case (stage 4 step 5).

In [ ]:
gap_table = gaps(components, runs, tuned)
crossing_table = crossings(gap_table)
save(gap_table, "gaps.csv")
save(crossing_table, "crossing.csv")
print(gap_table[["size", "comparison", "config", "error_gap", "se_error_gap", "within_2se", "bias_gap_rn", "variance_gap_rn"]]
      .to_string(index=False, float_format="{:.4g}".format))
print()
print("Provisional, on the sizes so far:")
print(crossing_table.to_string(index=False, float_format="{:.1f}".format))

## Item 5. Tuned settings and grid edges

The tuned setting of `rf` and of `gbm` at each size, and for each of its settings whether it sits at an edge of its grid. It does when its value is the first or the last of that setting's grid and its mean test deviance is lower than that of the configuration with the next value inside the grid (the model's other setting held at its tuned value) by more than twice the paired standard error. A tie is never an edge. The last column gives the step beyond the edge that the rule Grid edges fixed before part 1, or says that the grid cannot be extended on that side. Each grid is the one the inputs were made on. The rule has one round, read on the first run of part 1, and decision 19 applied it: on the report of a rerun, an edge on a side that could be extended is called a limit of the grid, with no step.

In [ ]:
tuned_table = tuned[["size", "model", "config", "rf_features", "rf_min_leaf", "gbm_depth", "gbm_trees", "mean", "se"]]
edge_table = edges(runs, summary, tuned, first_round=not RERUN)
save(tuned_table, "tuned.csv")
save(edge_table, "edges.csv")
print(tuned_table.to_string(index=False, float_format="{:.4g}".format))
print()
print(edge_table.drop(columns="neighbour_config").to_string(index=False, float_format="{:.4g}".format))

## Item 6. Where the error comes from

For each line, the deviance of the three groups of test rows (the largest 1% of test flows, the next 9% and the rest), averaged over draws. The three add up to the mean test deviance, so they can be compared across models; each group's share of the total is beside it.

In [ ]:
group_table = deviance_by_group(runs, lines)
save(group_table, "groups.csv")
print(group_table.drop(columns="config").to_string(index=False, float_format="{:.4g}".format))

## Item 7. Health

- The identity of section 8 in every row of the components: error = remainder_raw + variance_raw (`gaps` above also checked error gap = bias gap - variance gap, and would have stopped otherwise).
- For `ppml`, by size: the draws with an indicator constant in the sample (dropped from that fit), the fits scikit-learn warned had fallen back from Newton-Cholesky to L-BFGS, and the largest relative difference between the predicted and the observed training total.
- For every configuration, the draws whose test deviance is above that of `mean` on the same sample, with the largest test prediction divided by the largest training flow (`worse_than_mean.csv`).
- For `gbm`, the draws that diverge as in decision 9 (rule Stability): a test deviance more than ten times that of `mean` on the same sample, or a largest test prediction more than 1,000 times the largest training flow (`divergences.csv`).

These are reported, not acted on, except a divergence of `gbm` (see the rules at the end). `ppml` draws flagged here are part of the result.

In [ ]:
health = health_table(runs, components)
worse = worse_than_mean(runs)
diverged = divergences(runs)
print(f"Largest relative difference between error and remainder_raw + variance_raw: {health['identity_largest_difference'].max():.1e}")
assert health["identity_largest_difference"].max() <= 1e-8, "The identity of section 8 does not hold in the inputs."
save(health, "health.csv")
save(worse, "worse_than_mean.csv")
save(diverged, "divergences.csv")
print(health.to_string(index=False, float_format="{:.2g}".format))
print()
print("Draws worse than mean, by size and model (all configurations):")
print(worse.groupby(["size", "model"]).size().unstack(fill_value=0).to_string())

## Item 8. Time

Seconds per draw by size, summed over the fits of a draw (each `gbm` depth counted once, since one fit gives every tree count of its depth), with the seconds of the `gbm` fits, one per depth, beside them. Part 1 does not run the sizes above `CHECKPOINT_MAX`, so the minutes of the full run of stage 3 are projected: the seconds measured at the largest size of part 1, scaled by the ratios between sizes of section 9 of the plan (2,000 interpolated linearly between 1,000 and 5,000). Stages 5 and 6 are projected by scaling the minutes section 9 projected for them by how much slower or faster a draw at the largest size of part 1 ran here than in section 9 (the seconds measured here divided by section 9's seconds at that size). The last column assumes the rule Grid edges extends `gbm` to 1,000 trees, which about doubles the boosting time; on the report of a rerun, whose boosting already has 1,000 trees, it is not read. All of these are projections, on one worker. They rest on the seconds of the fits alone (stage 3 step 5): predicting the test rows and scoring them add to the time of a draw, so the projections probably understate it. They are not bounds, because the ratios of section 9 were measured on another machine and on a stand-in of the data. The timing draw of the full run (stage 3 step 3) times the fits and predictions of a draw together and replaces them.

In [ ]:
measured = seconds_per_draw(runs).set_index("size")
largest_part1 = measured.index.max()
projection = []
for n in SIZES:
    if n in measured.index:
        seconds, gbm_seconds = measured.loc[n, "seconds"], measured.loc[n, "gbm_seconds"]
    else:
        seconds = projected_seconds(measured.loc[largest_part1, "seconds"], largest_part1, n)
        gbm_seconds = projected_seconds(measured.loc[largest_part1, "gbm_seconds"], largest_part1, n)
    projection.append({"size": n, "measured": n in measured.index, "seconds_per_draw": seconds,
                       "gbm_seconds_per_draw": gbm_seconds, "stage3_minutes": DRAWS * seconds / 60,
                       "stage3_minutes_gbm_doubled": DRAWS * (seconds + gbm_seconds) / 60})
projection = pd.DataFrame(projection)

# Section 9 gave its minutes for draws as long as its own; scale them by how long
# a draw at the largest size of part 1 took here, against section 9's draw there.
scale = measured.loc[largest_part1, "seconds"] / section9_seconds(largest_part1)
scale_doubled = (measured.loc[largest_part1, "seconds"] + measured.loc[largest_part1, "gbm_seconds"]) / section9_seconds(largest_part1)
stages = [{"stage": "03_development.ipynb, full run", "minutes": projection["stage3_minutes"].sum(),
           "minutes_gbm_doubled": projection["stage3_minutes_gbm_doubled"].sum()}]
for name, minutes in SECTION9_STAGE_MINUTES.items():
    stages.append({"stage": name, "minutes": minutes * scale, "minutes_gbm_doubled": minutes * scale_doubled})
stages = pd.DataFrame(stages)
save(projection, "time.csv")
save(stages, "time_projection.csv")
print(projection.to_string(index=False, float_format="{:.1f}".format))
print()
print("Projected minutes on one worker:")
print(stages.to_string(index=False, float_format="{:.0f}".format))

## Item 9. Coefficients of `ppml`

By size, over the draws where it was estimated: the median, the 10th and 90th percentiles and the range of the intercept and of each coefficient, with the number of draws where its column was dropped because it was constant in the sample. Then, for each indicator, the number of draws in which it had no 1 and in which it had one to three 1s in the training sample. This item is read, not acted on: no rule of the checkpoint uses it (decisions 14 and 16).

In [ ]:
coefficients = ppml_coefficients(runs)
thin = thin_indicators(runs)
save(coefficients, "coefficients.csv")
save(thin, "thin_indicators.csv")
print(coefficients.to_string(index=False, float_format="{:.3g}".format))
print()
print(thin.to_string(index=False))

## What the rules of the checkpoint call for

The rules of section 6, read off the tables above as written. Claude posts them in the plan thread; Yu confirms each change, which goes into section 2 of the plan. A rule that fires is applied unless Yu writes a reason that does not concern which model is ahead.

- Grid edges: every tuned setting at an edge that can be extended. One round only, read on the first run of part 1.
- Stability: any divergence of `gbm` raises `GBM_MIN_LEAF` from 20 to 50. At n = 50 a leaf of 50 allows no split, so `gbm` there predicts the mean training flow.
- Sizes: if the provisional `best` comparison shows `one_crossing` (or `reversal_then_crossing`) between two sizes run so far, one size at the geometric mean of those two, rounded to the nearest 10.
- Ridge line (decision 15): Yu's choice; no result fires it.

On the report of a rerun, only item 7 is read: its flags are reported, and no rule fires (section 6, Grid edges: a new flag on the rerun changes nothing more).

In [ ]:
# Grid edges: the steps beyond every edge that can be extended. Not read on a rerun.
steps = []
for row in edge_table.itertuples():
    if row.at_edge and row.extension not in (CANNOT_EXTEND, GRID_LIMIT):
        step = f"{row.model} {row.setting}: {row.extension}"
        if step not in steps:
            steps.append(step)
grid_fires = len(steps) > 0 and not RERUN

# Stability: any divergence of gbm. On a rerun it is reported, not acted on.
stability_fires = len(diverged) > 0 and not RERUN

# Sizes: a crossing of the best comparison between two sizes run so far. Not read on a rerun.
best_crossing = crossing_table.set_index("comparison").loc["best"]
sizes_fires = best_crossing["pattern"] in ("one_crossing", "reversal_then_crossing") and not RERUN
sizes_call = ""
if sizes_fires:
    # The two sizes around the change that find_crossing interpolated.
    below, above = int(best_crossing["crossing_from"]), int(best_crossing["crossing_to"])
    new_size = int(round(np.sqrt(below * above), -1))
    sizes_call = f"add n = {new_size}, between {below} and {above} ({best_crossing['pattern']})"

rules = pd.DataFrame([
    {"rule": "Grid edges", "fires": grid_fires, "calls_for": "; ".join(steps) if grid_fires else ""},
    {"rule": "Stability", "fires": stability_fires,
     "calls_for": (f"GBM_MIN_LEAF 50 ({len(diverged)} gbm draw(s) diverged); at n = 50 a leaf of 50 allows no split, "
                   "so gbm there predicts the mean training flow") if stability_fires else ""},
    {"rule": "Sizes", "fires": sizes_fires, "calls_for": sizes_call},
    {"rule": "Ridge line", "fires": False, "calls_for": "Yu's choice (decision 15)"},
])
save(rules, "rules.csv")
print(rules.to_string(index=False))
print(f"\nWrote the report into {CHECKPOINT.relative_to(ROOT)}")